# Modelado — Trufi App Cochabamba
## CRISP-DM Fase 4 · Técnicas evaluadas bajo un mismo protocolo

**Pregunta:** ¿Cómo estimar el número esperado de consultas de ruta de Trufi App por celda H3 a partir de la
población, la ubicación y el contexto territorial de cada celda, mediante técnicas geoespaciales de ciencia de datos?

El protocolo (catálogo, variables, grilla, umbral de sobredispersión y regla de adopción) se declaró y se versionó
**antes** de esta ejecución: `reports/03_modelado/DECISIONES_03_modelado.md` (D-201 a D-207, commit M0).

| Tarea | Contenido | Salida |
|---|---|---|
| M1 | Cargar y separar prueba | `particion.csv` |
| M2 | Pliegues espaciales `GroupKFold(5)` por `block_id` | `folds.csv`, `figuras/pliegues.png` |
| M3 | Sobredispersión de M1 | `sobredispersion.csv` |
| M4 | Validación cruzada espacial de todo el catálogo | `cv_espacial.csv`, `resumen_cv.csv`, `oof_predicciones.parquet` |
| M5 | Ajuste anidado de M3 | `m3_grilla.csv` |
| M6 | Validación aleatoria y optimismo | `cv_aleatoria.csv`, `optimismo_aleatorio.csv` |
| M7 | Especificaciones alternativas | `especificaciones.csv` |
| M8 | Regla de adopción en código | `decision_adopcion.md`, `adopcion.json` |

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403
from trufi_ds import modeling as md
from trufi_ds.preparation import cell_polygons

import json
import geopandas as gpd
from sklearn.model_selection import GroupKFold, KFold

SEMILLA = 42
np.random.seed(SEMILLA)
SUB = "03_modelado"
crear_directorios(SUB, f"{SUB}/figuras")
REPORTE = PROJECT_ROOT / "reports" / SUB
PROCESSED = PROJECT_ROOT / "data" / "processed"
CATALOGO = {"B0": {}, "B1": {}, "M1": {}, "M2": {}, "M3": {}}
COLORES = {"B0": "#9E9E9E", "B1": "#795548", "M1": "#1976D2", "M2": "#388E3C", "M3": "#E65100"}
METRICAS = ["poisson_deviance", "d2", "mae", "rmse", "calibration", "spearman"]

## M1 · Cargar datos y separar prueba

Regla R1: de `test_blocks.csv` se lee solo la lista de `block_id` para descartar esas filas de inmediato. Ninguna
métrica, figura ni ajuste de este notebook usa filas de prueba.

In [3]:
tabla = pl.read_parquet(PROCESSED / "tabla_minable.parquet").filter(pl.col("in_model"))
bloques_prueba = pl.read_csv(PROCESSED / "test_blocks.csv")["block_id"]
es_prueba = pl.col("block_id").is_in(bloques_prueba.implode())

particion = (tabla.with_columns(pl.when(es_prueba).then(pl.lit("prueba")).otherwise(pl.lit("entrenamiento")).alias("conjunto"))
             .group_by("conjunto").agg(pl.col("block_id").n_unique().alias("bloques"), pl.len().alias("celdas"),
                                       pl.col("query_count").sum().alias("consultas"))
             .sort("conjunto"))
particion.write_csv(REPORTE / "particion.csv")

datos = tabla.filter(~es_prueba).sort("h3_cell").to_pandas()
del tabla  # a partir de aquí solo existen filas de entrenamiento
assert not set(datos["block_id"]) & set(bloques_prueba.to_list()), "Solapamiento de bloques"
print(f"Entrenamiento: {len(datos):,} celdas, {datos['block_id'].nunique()} bloques, {datos['query_count'].sum():,} consultas")
particion

Entrenamiento: 840 celdas, 35 bloques, 1,839,109 consultas


conjunto,bloques,celdas,consultas
str,u32,u32,i64
"""entrenamiento""",35,840,1839109
"""prueba""",8,247,79519


## M2 · Pliegues espaciales

`GroupKFold(5)` con `groups = block_id` (D-206): cada bloque H3 res 6 queda entero en un solo pliegue.

In [4]:
pliegues_esp = list(GroupKFold(5).split(datos, groups=datos["block_id"]))
fold_de = np.empty(len(datos), dtype=int)
for f, (_, va) in enumerate(pliegues_esp):
    fold_de[va] = f
folds = pd.DataFrame({"h3_cell": datos["h3_cell"], "block_id": datos["block_id"], "fold": fold_de})
folds.to_csv(REPORTE / "folds.csv", index=False)
resumen_folds = (folds.assign(q=datos["query_count"]).groupby("fold")
                 .agg(bloques=("block_id", "nunique"), celdas=("h3_cell", "size"), consultas=("q", "sum")))
resumen_folds.to_csv(REPORTE / "folds_resumen.csv")

gdf = gpd.GeoDataFrame(folds, geometry=cell_polygons(folds["h3_cell"].tolist()), crs="EPSG:4326")
fig, ax = plt.subplots(figsize=(11, 8))
gdf.plot(ax=ax, column=gdf["fold"].astype(str), categorical=True, cmap="Set2", legend=True, edgecolor="none",
         legend_kwds={"title": "pliegue", "loc": "upper right"})
ax.set_title(f"Pliegues espaciales GroupKFold(5) por bloque H3 res 6 — N={len(datos):,} celdas de entrenamiento")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); fig.tight_layout()
guardar_figura(fig, "pliegues", SUB); plt.close(fig)
resumen_folds

,bloques,celdas,consultas
fold,,,
0,7,168,61604
1,7,168,517926
2,6,169,1137525
3,8,168,76299
4,7,167,45755


## M3 · Sobredispersión (D-204)

χ² de Pearson / gl residuales de M1 en cada pliegue de entrenamiento. Umbral declarado: 1,5.

In [5]:
sobre = []
for f, (tr, _) in enumerate(pliegues_esp):
    entreno = datos.iloc[tr]
    m1 = md.M1().fit(entreno)
    sobre.append({"fold": f, "pearson_chi2_df": md.pearson_dispersion(m1, entreno), "n_train": len(tr)})
sobredispersion = pd.DataFrame(sobre)
phi_media = sobredispersion["pearson_chi2_df"].mean()
REFERENCIA_GLM = "M2" if phi_media > 1.5 else "M1"
sobredispersion.to_csv(REPORTE / "sobredispersion.csv", index=False)
print(f"φ medio = {phi_media:,.1f} → referencia interpretable: {REFERENCIA_GLM}")
sobredispersion

φ medio = 10,418.8 → referencia interpretable: M2


,fold,pearson_chi2_df,n_train
0,0,9417.525015,672
1,1,8986.763294,672
2,2,5560.439791,671
3,3,21224.337345,672
4,4,6904.837262,673


## M4–M5 · Validación cruzada espacial del catálogo (con ajuste anidado de M3)

Cada técnica se ajusta en el pliegue de entrenamiento y se evalúa en el de validación. M3 elige su configuración
con `GroupKFold(3)` interno (M5). Se guardan las predicciones fuera de pliegue.

In [6]:
cv_esp, oof_esp, busqueda_esp = md.cross_validate(datos, CATALOGO, pliegues_esp)
cv_esp.to_csv(REPORTE / "cv_espacial.csv", index=False)
oof_esp.to_parquet(REPORTE / "oof_predicciones.parquet", index=False)
m3_grilla = pd.DataFrame(busqueda_esp)
m3_grilla.to_csv(REPORTE / "m3_grilla.csv", index=False)
cv_esp[["model", "fold", *METRICAS, "warnings"]].round(4)

,model,fold,poisson_deviance,d2,mae,rmse,calibration,spearman,warnings
0,B0,0,2808.3245,-1.6628,1858.0906,2541.3676,5.9532,0.8980,
1,B1,0,478.4545,0.5463,334.5693,860.7072,1.3470,0.8826,
2,M1,0,645.2195,0.3882,472.3055,1156.4254,1.8468,0.8749,
3,M2,0,644.1167,0.3893,524.5400,1020.4899,2.0997,0.8967,
4,M3,0,370.1179,0.6491,323.9061,781.8836,1.3985,0.9031,
5,B0,1,4780.0095,0.5989,2665.0258,8351.0855,0.7272,0.8776,
6,B1,1,1519.2160,0.8725,1948.0115,6320.2862,1.2812,0.8531,
7,M1,1,1218.1796,0.8978,1449.3552,5634.6771,0.9388,0.8268,
8,M2,1,3026.4883,0.7460,1929.6890,7835.6202,0.5227,0.8802,
9,M3,1,1320.0365,0.8892,1644.0792,5498.7871,0.9236,0.8837,


In [7]:
def resumir(cv: pd.DataFrame) -> pd.DataFrame:
    dev = cv.pivot(index="fold", columns="model", values="poisson_deviance")
    mejor = dev.idxmin(axis=1)
    filas = []
    for m in [c for c in md.COMPLEXITY if c in dev.columns]:
        g = cv[cv["model"] == m]
        fila = {"model": m}
        for met in METRICAS:
            fila[f"{met}_mean"] = g[met].mean()
            fila[f"{met}_std"] = g[met].std(ddof=1)
        fila["folds_best_overall"] = int((mejor == m).sum())
        fila["folds_won_vs_B0"] = int((dev[m] < dev["B0"]).sum()) if m != "B0" else np.nan
        fila["folds_won_vs_B1"] = int((dev[m] < dev["B1"]).sum()) if m not in ("B0", "B1") else np.nan
        filas.append(fila)
    return pd.DataFrame(filas)

resumen_cv = resumir(cv_esp)
resumen_cv.to_csv(REPORTE / "resumen_cv.csv", index=False)
resumen_cv.round(4)

,model,poisson_deviance_mean,poisson_deviance_std,d2_mean,d2_std,mae_mean,mae_std,rmse_mean,rmse_std,calibration_mean,calibration_std,spearman_mean,spearman_std,folds_best_overall,folds_won_vs_B0,folds_won_vs_B1
0,B0,6579.3704,7821.3042,-0.3624,0.8465,2725.3340,1970.4620,6982.1074,7464.7467,3.3025,2.6330,0.8785,0.0174,0,NaN,NaN
1,B1,1585.9456,1589.7038,0.6574,0.1785,1434.1859,1600.9308,4631.3946,4983.9828,1.1203,0.2353,0.8695,0.0441,2,5.0,NaN
2,M1,2932.1042,3430.6898,0.3302,0.4671,1613.4571,2213.1489,5358.6891,6540.9226,0.7229,0.7291,0.8484,0.0659,1,5.0,1.0
3,M2,3853.0233,5773.6410,0.5224,0.1548,1694.6652,2176.3964,6098.9508,7487.3266,0.7893,0.7444,0.8993,0.0311,1,5.0,1.0
4,M3,3131.6311,5074.9545,0.5932,0.1838,1710.0514,2404.4354,5509.7163,7201.7416,0.8078,0.3647,0.8827,0.0333,1,5.0,3.0


**Métricas por anillo.** Se agrupan las predicciones fuera de pliegue por `distance_ring` para que el centro no
domine la lectura (riesgo 3).

In [8]:
anillo = dict(zip(datos["h3_cell"], datos["distance_ring"]))
por_anillo = []
for (m, a), g in oof_esp.assign(distance_ring=oof_esp["h3_cell"].map(anillo)).groupby(["model", "distance_ring"]):
    por_anillo.append({"model": m, "distance_ring": a, **md.metrics(g["y_true"], g["y_pred"])})
cv_por_anillo = pd.DataFrame(por_anillo)
cv_por_anillo.to_csv(REPORTE / "cv_por_anillo.csv", index=False)
cv_por_anillo.pivot(index="model", columns="distance_ring", values="poisson_deviance").round(2)

distance_ring,A1,A2,A3,A4
model,,,,
B0,14143.31,2375.65,1287.75,726.67
B1,3481.07,718.93,39.35,14.22
M1,6013.00,1904.29,85.23,30.36
M2,9189.64,807.92,24.60,9.03
M3,7351.19,788.82,55.91,29.47


In [9]:
dev = cv_esp.pivot(index="fold", columns="model", values="poisson_deviance")[md.COMPLEXITY]
fig, ax = plt.subplots(figsize=(8, 5))
for f in dev.index:
    ax.plot(range(len(dev.columns)), dev.loc[f], color="#BDBDBD", lw=1, zorder=1)
for i, m in enumerate(dev.columns):
    ax.scatter([i] * len(dev), dev[m], color=COLORES[m], s=40, zorder=2)
    ax.errorbar(i + 0.15, dev[m].mean(), yerr=dev[m].std(ddof=1), fmt="D", color="black", ms=5, capsize=4, zorder=3)
ax.set_xticks(range(len(dev.columns)), dev.columns); ax.set_yscale("log")
ax.set_ylabel("Devianza Poisson media (validación, log)")
ax.set_title(f"Devianza por pliegue espacial — N={len(datos):,} celdas, 5 pliegues\n(gris: un pliegue; ◆: media ± DE)")
fig.tight_layout(); guardar_figura(fig, "devianza_por_pliegue", SUB); plt.close(fig)

## M6 · Validación aleatoria y optimismo

Mismo universo y mismas técnicas con `KFold(5, shuffle=True, random_state=42)`. El optimismo es ΔD² = D² aleatorio
− D² espacial: cuánto sobrestima una validación que mezcla celdas vecinas entre entrenamiento y validación.

In [10]:
pliegues_ale = list(KFold(5, shuffle=True, random_state=SEMILLA).split(datos))
catalogo_ale = {**CATALOGO, "M3": {"inner_random": True}}
cv_ale, oof_ale, _ = md.cross_validate(datos, catalogo_ale, pliegues_ale)
cv_ale.to_csv(REPORTE / "cv_aleatoria.csv", index=False)

optimismo = (cv_esp.groupby("model")[["d2", "poisson_deviance"]].mean().add_suffix("_espacial")
             .join(cv_ale.groupby("model")[["d2", "poisson_deviance"]].mean().add_suffix("_aleatoria"))
             .reindex(md.COMPLEXITY))
optimismo["delta_d2"] = optimismo["d2_aleatoria"] - optimismo["d2_espacial"]
optimismo["ratio_devianza_espacial_aleatoria"] = optimismo["poisson_deviance_espacial"] / optimismo["poisson_deviance_aleatoria"]
optimismo.reset_index().to_csv(REPORTE / "optimismo_aleatorio.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(optimismo))
ax.bar(x - 0.2, optimismo["d2_espacial"], 0.4, label="Validación espacial (bloques)", color="#1976D2")
ax.bar(x + 0.2, optimismo["d2_aleatoria"], 0.4, label="Validación aleatoria", color="#90CAF9")
for i, d in enumerate(optimismo["delta_d2"]):
    ax.annotate(f"Δ={d:+.2f}", (i, max(optimismo["d2_espacial"].iloc[i], optimismo["d2_aleatoria"].iloc[i])),
                ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, optimismo.index); ax.set_ylabel("D² medio (5 pliegues)")
ax.set_title(f"Optimismo de la validación aleatoria — N={len(datos):,} celdas")
ax.legend(); fig.tight_layout(); guardar_figura(fig, "optimismo_aleatorio", SUB); plt.close(fig)
optimismo.round(4)

,d2_espacial,poisson_deviance_espacial,d2_aleatoria,poisson_deviance_aleatoria,delta_d2,ratio_devianza_espacial_aleatoria
model,,,,,,
B0,-0.3624,6579.3704,0.4587,5601.6344,0.8211,1.1745
B1,0.6574,1585.9456,0.9166,780.8550,0.2592,2.0310
M1,0.3302,2932.1042,0.8100,1739.4224,0.4798,1.6857
M2,0.5224,3853.0233,0.6931,3254.0370,0.1707,1.1841
M3,0.5932,3131.6311,0.8314,1786.4527,0.2382,1.7530


## M7 · Especificaciones alternativas (D-207)

(a) `log(population)` como covariable libre en M1 y M2; (b) objetivo `user_count` con todo el catálogo. Solo en
validación espacial, con los mismos pliegues.

In [11]:
cv_libre, _, _ = md.cross_validate(datos, {"M1_free": {"free_offset": True}, "M2_free": {"free_offset": True}}, pliegues_esp)
cv_usuarios, _, _ = md.cross_validate(datos, CATALOGO, pliegues_esp, target="user_count")

filas_esp = []
for base in ["M1", "M2"]:
    d_off = cv_esp[cv_esp["model"] == base].set_index("fold")["poisson_deviance"]
    d_lib = cv_libre[cv_libre["model"] == f"{base}_free"].set_index("fold")["poisson_deviance"]
    filas_esp.append({"especificacion": f"{base} con log(population) libre", "objetivo": "query_count",
                      "deviance_mean": d_lib.mean(), "deviance_std": d_lib.std(ddof=1),
                      "d2_mean": cv_libre[cv_libre["model"] == f"{base}_free"]["d2"].mean(),
                      "deviance_referencia": d_off.mean(), "mejora_pct_vs_offset": (1 - d_lib.mean() / d_off.mean()) * 100,
                      "pliegues_ganados_vs_offset": int((d_lib < d_off).sum())})
for m, g in cv_usuarios.groupby("model"):
    filas_esp.append({"especificacion": m, "objetivo": "user_count", "deviance_mean": g["poisson_deviance"].mean(),
                      "deviance_std": g["poisson_deviance"].std(ddof=1), "d2_mean": g["d2"].mean(),
                      "deviance_referencia": np.nan, "mejora_pct_vs_offset": np.nan, "pliegues_ganados_vs_offset": np.nan})
especificaciones = pd.DataFrame(filas_esp)
especificaciones.to_csv(REPORTE / "especificaciones.csv", index=False)
coef_libre = {m: md.make(m, free_offset=True).fit(datos).result_.params["log_population"] for m in ["M1", "M2"]}
adoptada_usuarios, _ = md.adoption_rule(cv_usuarios)
print("Coeficiente libre de log(population) (ajuste con todo el entrenamiento):", {k: round(v, 3) for k, v in coef_libre.items()})
print("Técnica que adoptaría la regla con user_count:", adoptada_usuarios)
especificaciones.round(4)

Coeficiente libre de log(population) (ajuste con todo el entrenamiento): {'M1': np.float64(1.156), 'M2': np.float64(1.171)}
Técnica que adoptaría la regla con user_count: B1


,especificacion,objetivo,deviance_mean,deviance_std,d2_mean,deviance_referencia,mejora_pct_vs_offset,pliegues_ganados_vs_offset
0,M1 con log(population) libre,query_count,2932.3390,3427.4245,0.3297,2932.1042,-0.0080,2.0
1,M2 con log(population) libre,query_count,3868.4908,5826.9575,0.5214,3853.0233,-0.4014,3.0
2,B0,user_count,1741.9504,1963.8715,-0.1882,NaN,NaN,NaN
3,B1,user_count,434.1544,370.1751,0.6710,NaN,NaN,NaN
4,M1,user_count,810.9683,875.5762,0.3797,NaN,NaN,NaN
5,M2,user_count,1012.3223,1441.1151,0.5404,NaN,NaN,NaN
6,M3,user_count,795.2643,1210.0696,0.6369,NaN,NaN,NaN


## M8 · Regla de adopción (D-205), aplicada en código

In [12]:
ADOPTADA, log_regla = md.adoption_rule(cv_esp)
log_regla.to_csv(REPORTE / "regla_adopcion_pasos.csv", index=False)

elegidas = m3_grilla[m3_grilla["chosen"]]
frecuencia = (elegidas.groupby(list(md.M3_GRID), dropna=False)
              .agg(veces=("fold", "size"), dev_interna=("inner_deviance_mean", "mean"))
              .reset_index().sort_values(["veces", "dev_interna"], ascending=[False, True]))
m3_config = {k: (None if pd.isna(v) else (int(v) if k != "learning_rate" else float(v)))
             for k, v in frecuencia.iloc[0][list(md.M3_GRID)].items()}

adopcion = {"adopted": ADOPTADA, "reference_glm": REFERENCIA_GLM, "overdispersion_phi_mean": phi_media,
            "m3_final_params": m3_config, "free_offset_coefficients": coef_libre,
            "spatial_folds": "GroupKFold(5) por block_id", "seed": SEMILLA}
(REPORTE / "adopcion.json").write_text(json.dumps(adopcion, indent=2), encoding="utf-8")

def fmt(m: str, met: str, nd: int = 3) -> str:
    r = resumen_cv.set_index("model").loc[m]
    return f"{r[f'{met}_mean']:.{nd}f} ± {r[f'{met}_std']:.{nd}f}"

tabla_md = ["| Técnica | Devianza | D² | MAE | Calibración | Spearman | Mejor en pliegues |", "|---|---|---|---|---|---|---|"]
for m in md.COMPLEXITY:
    tabla_md.append(f"| {m} | {fmt(m, 'poisson_deviance', 1)} | {fmt(m, 'd2')} | {fmt(m, 'mae', 1)} | "
                    f"{fmt(m, 'calibration')} | {fmt(m, 'spearman')} | {int(resumen_cv.set_index('model').loc[m, 'folds_best_overall'])}/5 |")
pasos_md = ["| Candidata | Frente a | Devianza candidata | Devianza referencia | Mejora | Pliegues ganados | Cumple | Adoptada tras el paso |",
            "|---|---|---|---|---|---|---|---|"]
for r in log_regla.itertuples():
    pasos_md.append(f"| {r.candidate} | {r.vs_best_simpler} | {r.deviance_candidate:.1f} | {r.deviance_best_simpler:.1f} | "
                    f"{r.improvement_pct:+.1f} % | {r.folds_won}/5 | {'sí' if r.passes else 'no'} | {r.adopted_after_step} |")

texto_resultado = (
    f"La técnica adoptada es **{ADOPTADA}**."
    + (" Ninguna técnica superó a B1: la información territorial disponible no mejora a la vecindad." if ADOPTADA == "B1" else "")
    + (" Ninguna técnica superó a la tasa global B0." if ADOPTADA == "B0" else "")
)
decision_md = f"""# Decisión de adopción (M8)

Generado por `notebooks/03_modelado.ipynb` con `trufi_ds.modeling.adoption_rule` (D-205). No se edita a mano.

**Regla:** se adopta la técnica más simple (B0 < B1 < M1 < M2 < M3) salvo que una más compleja reduzca la devianza
Poisson media de validación espacial en **más de 5 %** respecto de la mejor técnica más simple **y** le gane en
**al menos 4 de 5** pliegues.

## Resultado

{texto_resultado}

- Sobredispersión de M1: φ medio = {phi_media:,.1f} (umbral 1,5) → referencia interpretable: **{REFERENCIA_GLM}**.
- Configuración final de M3 (la más elegida en los 5 pliegues): `{m3_config}`.

## Validación cruzada espacial (media ± DE, 5 pliegues, N = {len(datos):,} celdas de entrenamiento)

{chr(10).join(tabla_md)}

## Aplicación paso a paso

{chr(10).join(pasos_md)}

## Optimismo de la validación aleatoria

| Técnica | D² espacial | D² aleatorio | ΔD² |
|---|---|---|---|
{chr(10).join(f"| {m} | {r.d2_espacial:.3f} | {r.d2_aleatoria:.3f} | {r.delta_d2:+.3f} |" for m, r in optimismo.iterrows())}

Fuentes: `cv_espacial.csv`, `resumen_cv.csv`, `regla_adopcion_pasos.csv`, `optimismo_aleatorio.csv`.
"""
(REPORTE / "decision_adopcion.md").write_text(decision_md, encoding="utf-8")
print(decision_md)

# Decisión de adopción (M8)

Generado por `notebooks/03_modelado.ipynb` con `trufi_ds.modeling.adoption_rule` (D-205). No se edita a mano.

**Regla:** se adopta la técnica más simple (B0 < B1 < M1 < M2 < M3) salvo que una más compleja reduzca la devianza
Poisson media de validación espacial en **más de 5 %** respecto de la mejor técnica más simple **y** le gane en
**al menos 4 de 5** pliegues.

## Resultado

La técnica adoptada es **B1**. Ninguna técnica superó a B1: la información territorial disponible no mejora a la vecindad.

- Sobredispersión de M1: φ medio = 10,418.8 (umbral 1,5) → referencia interpretable: **M2**.
- Configuración final de M3 (la más elegida en los 5 pliegues): `{'max_depth': None, 'min_samples_leaf': 20, 'learning_rate': 0.1}`.

## Validación cruzada espacial (media ± DE, 5 pliegues, N = 840 celdas de entrenamiento)

| Técnica | Devianza | D² | MAE | Calibración | Spearman | Mejor en pliegues |
|---|---|---|---|---|---|---|
| B0 | 6579.4 ± 7821.3 | -0.362 ± 0.84